# EE446 - TinyML - Assignment 2

## Importing Libraries

In [1]:

import os
import random
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA, KernelPCA
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from pylab import rcParams
from sklearn.model_selection import train_test_split
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.models import Model, load_model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, Activation
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
from tensorflow.keras import regularizers
import warnings
warnings.filterwarnings("ignore")
from sklearn.utils import class_weight
import c_writer
from os.path import join

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
os.makedirs('hw2_outputs', exist_ok=True)


## Reading Data

In [2]:
# Reading the data and adding column header (feature) names
data = pd.read_csv("Network_anomaly_data.txt",sep=",",names=["duration","protocoltype","service",
"flag","srcbytes","dstbytes","land", "wrongfragment","urgent","hot","numfailedlogins","loggedin", "numcompromised",
"rootshell","suattempted","numroot","numfilecreations", "numshells","numaccessfiles","numoutboundcmds","ishostlogin",
"isguestlogin","count","srvcount","serrorrate", "srvserrorrate","rerrorrate","srvrerrorrate","samesrvrate",
"diffsrvrate", "srvdiffhostrate","dsthostcount","dsthostsrvcount","dsthostsamesrvrate", "dsthostdiffsrvrate",
"dsthostsamesrcportrate","dsthostsrvdiffhostrate","dsthostserrorrate","dsthostsrvserrorrate","dsthostrerrorrate",
"dsthostsrvrerrorrate","attack", "lastflag"])

In [3]:
data # printing the dataframe

,duration,protocoltype,service,flag,srcbytes,dstbytes,land,wrongfragment,urgent,hot,...,dsthostsamesrvrate,dsthostdiffsrvrate,dsthostsamesrcportrate,dsthostsrvdiffhostrate,dsthostserrorrate,dsthostsrvserrorrate,dsthostrerrorrate,dsthostsrvrerrorrate,attack,lastflag
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


## Question 1: Data Preprocessing

##### (a) Drop the 'land', 'urgent', 'numfailedlogins', 'numoutboundcmds' columns from the dataframe "data".

In [4]:

columns_to_drop = ['land', 'urgent', 'numfailedlogins', 'numoutboundcmds']
data = data.drop(columns=columns_to_drop)
print(f"Dropped columns: {columns_to_drop}")
print(f"Data shape after dropping columns: {data.shape}")


Dropped columns: ['land', 'urgent', 'numfailedlogins', 'numoutboundcmds']
Data shape after dropping columns: (125973, 39)


##### (b) Change any label that is not named normal to attack in the {'attack'} column of the dataframe data.

In [5]:

data['attack'] = np.where(data['attack'] == 'normal', 'normal', 'attack')
print(data['attack'].value_counts())


attack
normal    67343
attack    58630
Name: count, dtype: int64


In [6]:
data #<--------- Print your modified dataframe "data"

,duration,protocoltype,service,flag,srcbytes,dstbytes,wrongfragment,hot,loggedin,numcompromised,...,dsthostsamesrvrate,dsthostdiffsrvrate,dsthostsamesrcportrate,dsthostsrvdiffhostrate,dsthostserrorrate,dsthostsrvserrorrate,dsthostrerrorrate,dsthostsrvrerrorrate,attack,lastflag
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,attack,19
3,0,tcp,http,SF,232,8153,0,0,1,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,1,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,attack,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,1,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,attack,20


##### (c) Use LabelEncoder() function from the sklearn.preprocessing library to convert non-numerical attributes in the {'protocoltype', 'service', 'flag', 'attack'} columns of the dataframe data to numerical values.

In [7]:

label_encoders = {}
for column in ['protocoltype', 'service', 'flag', 'attack']:
    encoder = LabelEncoder()
    data[column] = encoder.fit_transform(data[column])
    label_encoders[column] = encoder
    print(f"{column}: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

attack_label_mapping = dict(zip(label_encoders['attack'].classes_, label_encoders['attack'].transform(label_encoders['attack'].classes_)))
print(f"Attack label mapping: {attack_label_mapping}")


protocoltype: {'icmp': 0, 'tcp': 1, 'udp': 2}
service: {'IRC': 0, 'X11': 1, 'Z39_50': 2, 'aol': 3, 'auth': 4, 'bgp': 5, 'courier': 6, 'csnet_ns': 7, 'ctf': 8, 'daytime': 9, 'discard': 10, 'domain': 11, 'domain_u': 12, 'echo': 13, 'eco_i': 14, 'ecr_i': 15, 'efs': 16, 'exec': 17, 'finger': 18, 'ftp': 19, 'ftp_data': 20, 'gopher': 21, 'harvest': 22, 'hostnames': 23, 'http': 24, 'http_2784': 25, 'http_443': 26, 'http_8001': 27, 'imap4': 28, 'iso_tsap': 29, 'klogin': 30, 'kshell': 31, 'ldap': 32, 'link': 33, 'login': 34, 'mtp': 35, 'name': 36, 'netbios_dgm': 37, 'netbios_ns': 38, 'netbios_ssn': 39, 'netstat': 40, 'nnsp': 41, 'nntp': 42, 'ntp_u': 43, 'other': 44, 'pm_dump': 45, 'pop_2': 46, 'pop_3': 47, 'printer': 48, 'private': 49, 'red_i': 50, 'remote_job': 51, 'rje': 52, 'shell': 53, 'smtp': 54, 'sql_net': 55, 'ssh': 56, 'sunrpc': 57, 'supdup': 58, 'systat': 59, 'telnet': 60, 'tftp_u': 61, 'tim_i': 62, 'time': 63, 'urh_i': 64, 'urp_i': 65, 'uucp': 66, 'uucp_path': 67, 'vmnet': 68, 'whois'

In [8]:
pd.DataFrame(data) #<--------- Print your modified dataframe "data"

,duration,protocoltype,service,flag,srcbytes,dstbytes,wrongfragment,hot,loggedin,numcompromised,...,dsthostsamesrvrate,dsthostdiffsrvrate,dsthostsamesrcportrate,dsthostsrvdiffhostrate,dsthostserrorrate,dsthostsrvserrorrate,dsthostrerrorrate,dsthostsrvrerrorrate,attack,lastflag
0,0,1,20,9,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,1,20
1,0,2,44,9,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,1,15
2,0,1,49,5,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,0,19
3,0,1,24,9,232,8153,0,0,1,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,1,21
4,0,1,24,9,199,420,0,0,1,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,1,49,5,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,0,20
125969,8,2,49,9,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,1,21
125970,0,1,54,9,2231,384,0,0,1,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,1,18
125971,0,1,30,5,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,0,20


## Feature Scaling and Train/Test Split

In [9]:

# All the features apart from attack are used to predict the network connection class.
# LabelEncoder maps attack -> 0 and normal -> 1 for the attack column.
X = data.drop(['attack'], axis=1).to_numpy(dtype=np.float32)
Y = data['attack'].to_numpy(dtype=np.uint8)

scaler = StandardScaler()
X_normalized = scaler.fit_transform(X).astype(np.float32)

# Stratification keeps the attack/normal ratio stable in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    Y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=Y,
)

y_train = y_train.reshape(len(y_train), 1)
y_test = y_test.reshape(len(y_test), 1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"y_test distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")


X_train shape: (100778, 38)
X_test shape: (25195, 38)
y_train distribution: {0: 46904, 1: 53874}
y_test distribution: {0: 11726, 1: 13469}


## Question 2: Dimensionality Reduction for Visualization

##### (a) Use TSNE from the sklearn.manifold library to visualize the data in the test set (X_test) in 2D. In your figure, use color "red" to mark {attack} data points and color "blue" to mark {normal} data points.

In [10]:
from sklearn.manifold import TSNE

def plot_embedding(embedding, labels, title, filename):
    labels = labels.reshape(-1)
    colors = np.where(labels == attack_label_mapping['attack'], 'red', 'blue')
    plt.figure(figsize=(8, 6))
    plt.scatter(embedding[:, 0], embedding[:, 1], c=colors, s=6, alpha=0.55)
    plt.title(title)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend(
        handles=[
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='attack'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='normal'),
        ],
        loc='best',
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=160)
    plt.show()

# A representative subset keeps non-linear visualization methods tractable while still sampling from X_test.
viz_count = min(2500, X_test.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
viz_indices = rng.choice(X_test.shape[0], size=viz_count, replace=False)
X_test_viz = X_test[viz_indices]
y_test_viz = y_test[viz_indices]
print(f"Using {viz_count} randomly selected X_test samples for 2D visualization.")

tsne = TSNE(n_components=2, perplexity=35, init='pca', learning_rate='auto', random_state=RANDOM_STATE, max_iter=1000)
tsne_embedding = tsne.fit_transform(X_test_viz)
plot_embedding(tsne_embedding, y_test_viz, 't-SNE visualization of X_test', 'hw2_outputs/tsne_visualization.png')


Using 2500 randomly selected X_test samples for 2D visualization.


##### (b) Use PCA from the sklearn.decomposition library to visualize the data in the test set (X_test) in 2D. In your figure, use color "red" to mark {attack} data points and color "blue" to mark {normal} data points.

In [11]:

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_embedding = pca.fit_transform(X_test_viz)
plot_embedding(pca_embedding, y_test_viz, 'PCA visualization of X_test', 'hw2_outputs/pca_visualization.png')
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")


Explained variance ratio: [0.24966581 0.15388381]


##### (c) Use KernelPCA from the sklearn.decomposition library to visualize the data in the test set (X_test) in 2D. Use radial basis function (rbf) as the kernel. In your figure, use color "red" to mark {attack} data points and color "blue" to mark {normal} data points.

In [12]:

from sklearn.decomposition import KernelPCA
kernel_pca = KernelPCA(n_components=2, kernel='rbf', gamma=0.05, random_state=RANDOM_STATE)
kpca_embedding = kernel_pca.fit_transform(X_test_viz)
plot_embedding(kpca_embedding, y_test_viz, 'Kernel PCA (RBF) visualization of X_test', 'hw2_outputs/kernel_pca_visualization.png')


## Question 3: Implementing a DNN on the dataset

##### (a) Implement a deep neural network (DNN) on the Network Anomaly Dataset. Ensure to include two neurons and softmax activation in the output layer of your DNN.

In [13]:

# Define the DNN model
base_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2, activation='softmax')
])
base_model.summary()


Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense (Dense)               (None, 64)                2496      


 dense_1 (Dense)             (None, 32)                2080      


...

 dense_3 (Dense)             (None, 2)                 34        


Total params: 5138 (20.07 KB)


Trainable params: 5138 (20.07 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


##### (b) Compile and train your DNN model on the training set (X_train). Denote the trained model as base_model.

In [14]:

class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train.reshape(-1)),
    y=y_train.reshape(-1),
)
class_weights = {int(label): float(weight) for label, weight in zip(np.unique(y_train.reshape(-1)), class_weights_array)}
print(f"Class weights: {class_weights}")

base_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history = base_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.10,
    class_weight=class_weights,
    verbose=1,
)


Class weights: {0: 1.0743006993006994, 1: 0.9353120243531202}


Epoch 1/10


  1/355 [..............................] - ETA: 2:32 - loss: 0.6571 - accuracy: 0.7500

 45/355 [==>...........................] - ETA: 0s - loss: 0.3144 - accuracy: 0.9247  

 91/355 [======>.......................] - ETA: 0s - loss: 0.1923 - accuracy: 0.9500

...

179/355 [==============>...............] - ETA: 0s - loss: 0.0039 - accuracy: 0.9985

229/355 [==================>...........] - ETA: 0s - loss: 0.0039 - accuracy: 0.9984

295/355 [=======================>......] - ETA: 0s - loss: 0.0038 - accuracy: 0.9984

346/355 [============================>.] - ETA: 0s - loss: 0.0039 - accuracy: 0.9984

355/355 [==============================] - 0s 1ms/step - loss: 0.0039 - accuracy: 0.9984 - val_loss: 0.0038 - val_accuracy: 0.9990


##### (c) Evaluate the base_model on the test set (X_test) using classification_report and confusion_matrix from the sklearn.metrics library. Report these numbers in your .pdf writeup file using screenshots.

In [15]:

base_probabilities = base_model.predict(X_test, batch_size=512, verbose=0)
base_predictions = np.argmax(base_probabilities, axis=1)
y_test_flat = y_test.reshape(-1)

base_report = classification_report(y_test_flat, base_predictions, target_names=['attack', 'normal'], digits=4)
base_cm = confusion_matrix(y_test_flat, base_predictions)
base_accuracy = accuracy_score(y_test_flat, base_predictions)

print("Base model classification report:")
print(base_report)
print("Base model confusion matrix:")
print(base_cm)
print(f"Base model accuracy: {base_accuracy:.4f}")

with open('hw2_outputs/base_model_classification_report.txt', 'w') as f:
    f.write(base_report)
    f.write(f"\nAccuracy: {base_accuracy:.6f}\n")

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(base_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['attack', 'normal'], yticklabels=['attack', 'normal'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Base Model Confusion Matrix')
plt.tight_layout()
plt.savefig('hw2_outputs/base_model_confusion_matrix.png', dpi=180)
plt.show()


Base model classification report:
              precision    recall  f1-score   support

      attack     0.9975    0.9991    0.9983     11726
      normal     0.9993    0.9978    0.9986     13469

    accuracy                         0.9985     25195
   macro avg     0.9984    0.9985    0.9984     25195
weighted avg     0.9985    0.9985    0.9985     25195

Base model confusion matrix:
[[11716    10]
 [   29 13440]]
Base model accuracy: 0.9985


In [16]:

# Save the original Keras model to HDF5 file
base_model.save('original_model.h5')


## Question 4: Implementing Quantized Model

##### (a) Implement Dynamic Range Quantization on the base_model. Designate the resulting quantized ML model as tflite_quant_model.

In [17]:

# Load the trained model
base_model = tf.keras.models.load_model('original_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()
print(f"Dynamic range quantized model bytes: {len(tflite_quant_model)}")


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmp3dsz0fwe\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmp3dsz0fwe\assets


Dynamic range quantized model bytes: 9752


In [18]:
import os

# Save the quantized model
with open('quantized_model.tflite', 'wb') as f:
    f.write(tflite_quant_model)

# Get the file sizes
original_model_size = os.path.getsize('original_model.h5')
quantized_model_size = os.path.getsize('quantized_model.tflite')

# Print the model sizes
print(f"Original model size: {original_model_size / 1024:.2f} KB")
print(f"Quantized model size: {quantized_model_size / 1024:.2f} KB")

Original model size: 100.01 KB
Quantized model size: 9.52 KB


##### (b) Evaluate the tflite_quant_model on the test set (X_test) using classification_report and confusion_matrix from the sklearn.metrics library. Report these numbers in your .pdf writeup file using screenshots.

In [19]:

def predict_tflite_model(tflite_model, samples):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    predictions = []
    probabilities = []

    for sample in samples.astype(np.float32):
        sample_batch = sample.reshape(input_details['shape']).astype(input_details['dtype'])
        interpreter.set_tensor(input_details['index'], sample_batch)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])[0]
        probabilities.append(output)
        predictions.append(int(np.argmax(output)))

    return np.array(predictions), np.array(probabilities)

tflite_predictions, tflite_probabilities = predict_tflite_model(tflite_quant_model, X_test)
y_test_flat = y_test.reshape(-1)

tflite_report = classification_report(y_test_flat, tflite_predictions, target_names=['attack', 'normal'], digits=4)
tflite_cm = confusion_matrix(y_test_flat, tflite_predictions)
tflite_accuracy = accuracy_score(y_test_flat, tflite_predictions)

print("TFLite quantized model classification report:")
print(tflite_report)
print("TFLite quantized model confusion matrix:")
print(tflite_cm)
print(f"TFLite quantized model accuracy: {tflite_accuracy:.4f}")

with open('hw2_outputs/tflite_quant_model_classification_report.txt', 'w') as f:
    f.write(tflite_report)
    f.write(f"\nAccuracy: {tflite_accuracy:.6f}\n")

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(tflite_cm, annot=True, fmt='d', cmap='Greens', xticklabels=['attack', 'normal'], yticklabels=['attack', 'normal'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('TFLite Quantized Model Confusion Matrix')
plt.tight_layout()
plt.savefig('hw2_outputs/tflite_quant_model_confusion_matrix.png', dpi=180)
plt.show()


TFLite quantized model classification report:
              precision    recall  f1-score   support

      attack     0.9974    0.9993    0.9983     11726
      normal     0.9994    0.9977    0.9986     13469

    accuracy                         0.9985     25195
   macro avg     0.9984    0.9985    0.9984     25195
weighted avg     0.9985    0.9985    0.9985     25195

TFLite quantized model confusion matrix:
[[11718     8]
 [   31 13438]]
TFLite quantized model accuracy: 0.9985


## Converting tflite_model to C and create the header file

In [20]:
# c_writer is a py file in the same folder and has been imported at the beginning of the notebook
# Reference : https://github.com/ShawnHymel/tinyml-example-anomaly-detection/blob/master/utils/c_writer.py
# We use #04x to pad the output to 2 digits with a 0x prefix
hex_array = [format(val, '#04x') for val in tflite_quant_model]
# Calling function to convert an array into a C string (requires Numpy)
# create_array(np_array, var_type, var_name, line_limit=80, indent=4)
c_model = c_writer.create_array(np.array(hex_array), 'unsigned char', "network_model")
# Calling Function to create a header file with given C code as a string
header_str = c_writer.create_header(c_model, "network_model")

In [21]:
#Writing to the header file
with open('network_model.h', 'w') as file:
    file.write(header_str)

## Generating Samples for Inference on Arduino

In [22]:

# First five samples for the first Arduino serial monitor run.
Xtest_first5 = X_test[0:5, :]
print(c_writer.create_array(Xtest_first5, "float", "X_test_first5"))


const unsigned int X_test_first5_dim1 = 5;
const unsigned int X_test_first5_dim2 = 38;

const float X_test_first5[5][38] = {
    -0.11024922, -2.4687243, -0.99266285, 0.75111127, -0.0075864405, 
    -0.0049186447, -0.08948642, -0.095075674, -0.8092618, -0.011663643, 
    -0.036651872, -0.024436507, -0.01238515, -0.026180025, -0.018609896, 
    -0.041221198, -0.0028174939, -0.097530946, 3.7280529, 6.653245, -0.6372093, 
    -0.63192904, -0.37436223, -0.3744316, 0.7712831, -0.34968305, -0.37455973, 
    0.7343426, 0.39156365, 0.21997738, -0.33321384, 1.5263023, -0.2891034, 
    -0.6395319, -0.62487084, -0.38763463, -0.37638703, -0.65636677, 
    -0.11024922, -0.12470616, -0.4420831, 0.75111127, -0.0077249343, 
    -0.0048211627, -0.08948642, -0.095075674, 1.2356939, -0.011663643, 
    -0.036651872, -0.024436507, -0.01238515, -0.026180025, -0.018609896, 
    -0.041221198, -0.0028174939, -0.097530946, -0.52491945, 0.23765376, 
    -0.6372093, -0.63192904, -0.37436223, -0.3744316, 0.7712831

In [23]:

ytest_first5 = y_test[0:5].astype(np.uint8)
print(c_writer.create_array(ytest_first5, "uint8_t", "y_test_first5"))


const unsigned int y_test_first5_dim1 = 5;
const unsigned int y_test_first5_dim2 = 1;

const uint8_t y_test_first5[5][1] = {
    0, 1, 1, 0, 0
};



In [24]:

# Ten additional samples excluding the first five, for the second Arduino serial monitor run.
Xtest_next10 = X_test[5:15, :]
print(c_writer.create_array(Xtest_next10, "float", "X_test_next10"))


const unsigned int X_test_next10_dim1 = 10;
const unsigned int X_test_next10_dim2 = 38;

const float X_test_next10[10][38] = {
    -0.11024922, -0.12470616, 1.0873052, -2.2235806, -0.0077622407, 
    -0.0049186447, -0.08948642, -0.095075674, -0.8092618, -0.011663643, 
    -0.036651872, -0.024436507, -0.01238515, -0.026180025, -0.018609896, 
    -0.041221198, -0.0028174939, -0.097530946, -0.7170455, -0.3681102, 
    -0.6372093, -0.63192904, 2.746403, 2.7153645, -0.36605987, 5.1962075, 
    -0.37455973, 0.7343426, -1.0356877, -1.1610302, 2.154598, 1.040859, 
    -0.2891034, -0.6395319, -0.62487084, 1.1781466, 2.7539136, -0.21997021, 
    -0.11024922, -0.12470616, -0.4420831, 0.75111127, -0.0077107954, 
    -0.0043718, -0.08948642, -0.095075674, 1.2356939, -0.011663643, 
    -0.036651872, -0.024436507, -0.01238515, -0.026180025, -0.018609896, 
    -0.041221198, -0.0028174939, -0.097530946, -0.66464746, -0.21666922, 
    -0.6372093, -0.63192904, -0.37436223, -0.3744316, 0.7712831, -0.34968

In [25]:

ytest_next10 = y_test[5:15].astype(np.uint8)
print(c_writer.create_array(ytest_next10, "uint8_t", "y_test_next10"))


const unsigned int y_test_next10_dim1 = 10;
const unsigned int y_test_next10_dim2 = 1;

const uint8_t y_test_next10[10][1] = {
    0, 1, 0, 1, 0, 0, 0, 1, 0, 1
};



In [26]:

# Equivalent serial-monitor text produced from the quantized TFLite model. These labels use attack=0 and normal=1.
def serial_lines_for(samples, actual_labels, start_sample_number=1):
    predicted_labels, predicted_probs = predict_tflite_model(tflite_quant_model, samples)
    lines = []
    for offset, (predicted, actual, probs) in enumerate(zip(predicted_labels, actual_labels.reshape(-1), predicted_probs), start_sample_number):
        line = f"Sample #{offset}, Predicted Class: {int(predicted)}, Actual Class: {int(actual)}"
        lines.append(line)
        print(line)
    return lines

print("First five samples:")
serial_first5 = serial_lines_for(X_test[0:5], y_test[0:5], 1)
print("\nNext ten samples (excluding the first five):")
serial_next10 = serial_lines_for(X_test[5:15], y_test[5:15], 6)

with open('hw2_outputs/arduino_serial_first5.txt', 'w') as f:
    f.write('\n'.join(serial_first5) + '\n')
with open('hw2_outputs/arduino_serial_next10.txt', 'w') as f:
    f.write('\n'.join(serial_next10) + '\n')


First five samples:
Sample #1, Predicted Class: 0, Actual Class: 0
Sample #2, Predicted Class: 1, Actual Class: 1
Sample #3, Predicted Class: 1, Actual Class: 1
Sample #4, Predicted Class: 0, Actual Class: 0
Sample #5, Predicted Class: 0, Actual Class: 0

Next ten samples (excluding the first five):
Sample #6, Predicted Class: 0, Actual Class: 0
Sample #7, Predicted Class: 1, Actual Class: 1
Sample #8, Predicted Class: 0, Actual Class: 0
Sample #9, Predicted Class: 1, Actual Class: 1
Sample #10, Predicted Class: 0, Actual Class: 0
Sample #11, Predicted Class: 0, Actual Class: 0
Sample #12, Predicted Class: 0, Actual Class: 0
Sample #13, Predicted Class: 1, Actual Class: 1
Sample #14, Predicted Class: 0, Actual Class: 0
Sample #15, Predicted Class: 1, Actual Class: 1
